In [0]:
import requests
import json

In [0]:
def get_slack_webhook():
    return dbutils.secrets.get(scope="compliance-grc", key="slack-webhook")

def notify_slack_incident(
    test_name: str,
    description: str,
    responsible_area: str,
    incident_count: int,
    threshold: int,
    risco_id: str,
    output_table: str
):
    webhook_url = get_slack_webhook()

    payload = {
        "blocks": [
            {
                "type": "section",
                "text": {"type": "mrkdwn", "text": ":warning: *GRC Automated Test – Incidente Encontrado*"}
            },
            {
                "type": "section",
                "text": {"type": "mrkdwn", "text": f"*Nome do Teste:*\n`{test_name}`"}
            },
            {"type": "divider"},
            {
                "type": "section",
                "fields": [
                    {"type": "mrkdwn", "text": f"*Área Responsável:*\n{responsible_area}"},
                    {"type": "mrkdwn", "text": f"*ID do Risco:*\n{risco_id}"}
                ]
            },
            {
                "type": "section",
                "text": {"type": "mrkdwn", "text": f"*Tabela de Incidências:*\n`compliance.continuous_audit.{output_table}`"}
            },
            {"type": "divider"},
            {
                "type": "section",
                "text": {"type": "mrkdwn", "text": f"*Descrição do Teste:*\n{description}"}
            },
            {"type": "divider"},
            {
                "type": "section",
                "fields": [
                    {"type": "mrkdwn", "text": f"*Threshold:*\n{threshold}"},
                    {"type": "mrkdwn", "text": f"*Incidentes:*\n{incident_count}"}
                ]
            },
            {"type": "divider"},
            {
                "type": "context",
                "elements": [
                    {
                        "type": "mrkdwn",
                        "text": ":point_right::skin-tone-2: <https://cercbr.sharepoint.com/sites/GovernancaCERC/Lists/CONTINUOUS_AUDIT_TRIGGER_DETAILS/AllItems.aspx|Clique aqui> para checar os cards e atualizar os assignees."
                    }
                ]
		    }
        ]
    }

    response = requests.post(webhook_url, json=payload)
    if response.status_code != 200:
        raise Exception(f"Failed to send Slack message: {response.status_code} {response.text}")